Finetuning Qwen 2.5

In [ ]:
# ================================================================
# QWEN2.5-VL-7B QLoRA FINETUNING
# (FAST UNZIP VERSION)
# ================================================================


# ================================================================
# 1️⃣ INSTALL DEPENDENCIES (RUN ONCE)
# ================================================================
# Uncomment first run only
# !pip install -U pip
# !pip install -U bitsandbytes accelerate datasets peft trl
# !pip install -U hf_transfer pillow scikit-learn gdown
# !pip install -U git+https://github.com/huggingface/transformers.git


# ================================================================
# 2️⃣ IMPORTS
# ================================================================
import os
import torch
import gdown

from PIL import Image
from datasets import load_dataset

from transformers import (
    AutoModelForImageTextToText,
    AutoProcessor,
    BitsAndBytesConfig
)

from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer
from huggingface_hub import login


# ================================================================
# 3️⃣ CONFIG
# ================================================================
HF_TOKEN = "YOUR_HF_TOKEN"
HF_USERNAME = "YOUR_HF_USERNAME"

MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"

GDRIVE_FILE_ID = "data_location"

ZIP_PATH = "/workspace/dataset.zip"
DATA_PATH = "/workspace/Training_Dataset"

OUTPUT_DIR = "./qwen2.5vl_lora"

os.makedirs(DATA_PATH, exist_ok=True)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"


# ================================================================
# 4️⃣ LOGIN TO HF
# ================================================================
login(HF_TOKEN)


# ================================================================
# 5️⃣ DOWNLOAD DATASET (GOOGLE DRIVE)
# ================================================================
print("Downloading dataset...")

url = f"gdrive_url"
gdown.download(url, ZIP_PATH, quiet=False)

print("Download complete.")


# ================================================================
# 6️⃣ ⚡ FAST UNZIP (LINUX UNZIP — MUCH FASTER)
# ================================================================
print("Extracting dataset (fast unzip)...")

!mkdir -p /workspace/dataset
!unzip -q /workspace/dataset.zip -d /workspace/dataset

print("Extraction done.")

# check structure
!ls /workspace/MINISTRAL_DATASET


# OPTIONAL: remove zip to save space
# !rm /workspace/dataset.zip


# ================================================================
# 7️⃣ LOAD QWEN2.5-VL-7B (4-BIT)
# ================================================================
print("Loading model...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(MODEL_NAME)

print("Model loaded.")


# ================================================================
# 8️⃣ ADD LORA ADAPTERS (SAFE TARGETS)
# ================================================================
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q_proj","k_proj","v_proj","o_proj"
    ],
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


# ================================================================
# 9️⃣ LOAD DATASET
# ================================================================
dataset = load_dataset(
    "json",
    data_files={
        "train": f"{DATA_PATH}/train/train.jsonl",
        "validation": f"{DATA_PATH}/val/val.jsonl"
    }
)

print(dataset)


# ================================================================
# 🔟 FORMAT FUNCTION (VERY IMPORTANT)
# ================================================================
def format_example(example):

    messages = example["messages"]

    split = "val" if "val" in example.get("__source_file__", "") else "train"

    image_path = os.path.join(
        DATA_PATH,
        split,
        example["images"][0]
    )

    image = Image.open(image_path).convert("RGB")

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    model_inputs = processor(
        text=text,
        images=image,
        return_tensors="pt"
    )

    # remove batch dimension
    model_inputs = {k: v.squeeze(0) for k, v in model_inputs.items()}

    model_inputs["labels"] = model_inputs["input_ids"].clone()

    return model_inputs


# ================================================================
# 1️⃣1️⃣ TRAINING CONFIG (STABLE FOR 7B)
# ================================================================
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    num_train_epochs=1,

    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,

    learning_rate=1e-5,

    bf16=True,
    gradient_checkpointing=True,

    logging_steps=10,

    eval_strategy="steps",
    eval_steps=1000,

    save_strategy="steps",
    save_steps=1000,

    warmup_ratio=0.05,
)


# ================================================================
# 1️⃣2️⃣ TRAINER
# ================================================================
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    formatting_func=format_example,
    args=training_args,
)


# ================================================================
# 1️⃣3️⃣ TRAIN
# ================================================================
print("Starting training...")
trainer.train()
print("Training complete.")


# ================================================================
# 1️⃣4️⃣ SAVE FINAL ADAPTER
# ================================================================
FINAL_ADAPTER_PATH = "./qwen2.5vl_lora_final"

model.save_pretrained(FINAL_ADAPTER_PATH)
processor.save_pretrained(FINAL_ADAPTER_PATH)

print("Adapter saved.")


# ================================================================
# 1️⃣5️⃣ PUSH ADAPTER TO HUGGING FACE
# ================================================================
REPO_ID = f"{HF_USERNAME}/qwen2.5vl-lightcurve-lora"

print("Uploading to HF...")

model.push_to_hub(REPO_ID)
processor.push_to_hub(REPO_ID)

print("✅ Uploaded:", REPO_ID)
